# Mimicore cloud voice training

Trains your **Mimicore** voice on a free Google GPU, using **your own** Google account and Drive.

1. Press **▶** on the cell below (or *Runtime > Run all*).
2. Click **Run anyway**, then allow **Google Drive** access.
3. Keep this tab open. The Mimicore app shows the progress and installs the voice when it's done.

**Your data:** the recordings are in your Drive (`Mimicore Cloud Training` folder) and this temporary Colab machine,
which Google wipes when the session ends. Nothing is sent to Ribanix Labs. Recordings and checkpoints are deleted from
your Drive when the voice is installed, unless you chose to keep them.


In [ ]:
#@title ▶ Train my Mimicore voice — enhanced
#@markdown Start a cloud training job in Mimicore, then run this cell in a **GPU** runtime.
#@markdown Keep the tab open. If interrupted, rerun to restore the latest verified backup.
#@markdown Drive mounting grants broad Drive access. This wrapper uses `Mimicore Cloud Training`; downloaded training code also runs in this session.

import collections
import contextlib
import hashlib
import html
import json
import os
from pathlib import Path
import re
import shutil
import signal
import subprocess
import sys
import tarfile
import tempfile
import threading
import time
import traceback
import urllib.request
import uuid

os.environ["MPLBACKEND"] = "Agg"  # Force headless matplotlib backend (prevents Colab inline backend ValueError)

# Defaults retain the original model/quality choices. These improve reliability,
# not the acoustic model. Training time depends on the audio, GPU and epoch count.
ROOT_NAME = "Mimicore Cloud Training"
APPLIO_COMMIT = "821af89b8716a27c612b1b107db65e1cf56033c2"
MIMICORE_COMMIT = "2.9.6"
REPO_RAW = "https://raw.githubusercontent.com/ribanixlabs/mimicore-app/main/colab"
# The TITAN base is a PyTorch checkpoint (a pickle): it is pinned to one commit of the Hugging Face repo and verified by exact size + SHA-256
# before it is ever loaded (fail closed). `main` is never fetched.
TITAN_COMMIT = "cb72bb5bb05a85f07b17a33b014b694eaa9525f7"
TITAN_URL = "https://huggingface.co/blaise-tk/TITAN/resolve/" + TITAN_COMMIT + "/models/medium/48k/pretrained/{}-f048k-TITAN-Medium.pth"
TITAN_FILES = {
    "G": (452338845, "51a6dc93687f7e1a6051be3dec958289958c9c738d5c520e4fa956e7886ee153"),
    "D": (857126469, "4999a5b66aec0a9ab7e845063eeb242e47a611ab0e6260fb0b5ca44a7c5bbc44"),
}
HOME = Path("/content/mimicore-enhanced")
APPLIO, TOOLS = HOME / "applio", HOME / "tools"
WORK = Path("/content/mimicore-work")
PYTHON = HOME / "venv/bin/python"
BACKUP_SECONDS = 300
MAX_OOM_RETRIES = 2
REMOVE_RECORDINGS_AFTER_SUCCESS = False
RELEASE_GPU_AFTER_SUCCESS = True
AUDIO_EXTENSIONS = {".wav", ".flac", ".mp3", ".m4a", ".ogg", ".opus", ".aac", ".aiff", ".aif", ".wma", ".webm", ".mp4"}
TERMINAL = {"cancelled", "installed", "done"}
OOM_TEXT = ("out of memory", "out of graphics memory", "bad allocation", "memory allocation failure")


class Cancelled(Exception):
    pass


def banner(message, color="#7c6cf0"):
    from IPython.display import HTML, display
    display(HTML(f'<div style="padding:12px;border-radius:10px;background:{color};color:white;font:600 15px system-ui">{html.escape(str(message))}</div>'))


def read_json(path, attempts=4):
    # Drive files can be observed during an in-place write. Never turn a failed
    # read into an empty status and overwrite a cancellation/installation.
    for attempt in range(attempts):
        try:
            value = json.loads(path.read_text(encoding="utf-8"))
            if not isinstance(value, dict):
                raise ValueError(f"{path.name} must contain a JSON object")
            return value
        except (OSError, ValueError):
            if attempt == attempts - 1:
                raise
            time.sleep(0.25 * (attempt + 1))


def digest(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(8 << 20), b""):
            h.update(block)
    return h.hexdigest()


def integer(value, name, low, high):
    if isinstance(value, bool) or not re.fullmatch(r"[0-9]+", str(value)):
        raise ValueError(f"{name} must be a whole number")
    result = int(value)
    if not low <= result <= high:
        raise ValueError(f"{name} must be between {low} and {high}")
    return result


class Job:
    def __init__(self, folder):
        self.folder = Path(folder)
        self.spec = read_json(self.folder / "job.json")
        self.status_file = self.folder / "status.json"
        self.status = read_json(self.status_file)
        self.lock = threading.RLock()
        self.dirty = {}
        self.last_check = 0.0
        self.epochs = integer(self.spec.get("epochs", 250), "epochs", 1, 10000)
        raw = str(self.spec.get("id") or "voice")
        slug = re.sub(r"[^a-z0-9_-]+", "_", raw.lower()).strip("_")[:48] or "voice"
        # Folder identity prevents collisions between sanitized IDs and jobs.
        suffix = hashlib.sha256((self.folder.name + "\0" + raw).encode()).hexdigest()[:12]
        self.name = f"{slug}_{suffix}"
        self.base = self.spec.get("base") or self.spec.get("basis") or "titan"
        if self.base not in ("titan", "rvcv2"):
            raise ValueError("Unknown base model; expected titan or rvcv2")
        self.batch = self.spec.get("batch")
        if self.batch:
            self.batch = integer(self.batch, "batch", 1, 128)

    def check(self, every=0):
        # A Drive read that fails is NOT a stop request: only a successful read showing cancel/terminal stops the job
        # (a transient FUSE error must never kill hours of training).
        now = time.monotonic()
        if every and now - self.last_check < every:
            return
        self.last_check = now
        try:
            spec = read_json(self.folder / "job.json")
            state = read_json(self.status_file).get("state")
        except (OSError, ValueError):
            return
        if spec.get("cancel") or state in TERMINAL:
            raise Cancelled(f"Job stopped ({state or 'cancel requested'}).")

    def update(self, **fields):
        with self.lock:
            self.dirty.update(fields)
            self.status.update(fields)

    def write(self, **fields):
        # Keep existing Drive file IDs. This lock protects notebook threads only;
        # Drive mount read/merge/write is NOT a cross-client transaction.
        with self.lock:
            disk = read_json(self.status_file)
            if disk.get("state") in TERMINAL:
                self.status = disk
                self.dirty.clear()
                return False
            self.dirty.update(fields)
            disk.update(self.dirty)
            disk["updated"] = int(time.time())
            data = json.dumps(disk, ensure_ascii=False).encode("utf-8")
            with self.status_file.open("r+b") as f:
                f.seek(0)
                f.write(data)
                f.truncate()
                f.flush()
                os.fsync(f.fileno())
            self.status = disk
            self.dirty.clear()
            return True

    def try_write(self, **fields):
        # Progress updates must never kill a long training run because Drive hiccuped: keep the values for the next
        # heartbeat instead. (Not for the final done/uploading writes: those must succeed or raise.)
        try:
            return self.write(**fields)
        except (OSError, ValueError) as exc:
            self.update(**fields)
            print(f"Drive status update delayed: {exc}")
            return False


def heartbeat(job, stop):
    while not stop.wait(45):
        try:
            job.write()
        except Exception as exc:
            print(f"Drive heartbeat unavailable: {exc}")


def find_job():
    from google.colab import drive
    if not Path("/content/drive/MyDrive").is_dir():
        drive.mount("/content/drive")
    root = Path("/content/drive/MyDrive") / ROOT_NAME
    jobs = []
    for path in root.glob("*/job.json"):
        try:
            job = Job(path.parent)
            if job.status.get("state") not in TERMINAL and not job.spec.get("cancel"):
                if not (job.folder / "voice.tar").is_file():
                    raise ValueError("Missing app-created voice.tar placeholder; recreate this job in Mimicore")
                created = float(job.spec.get("created") or 0)
                jobs.append((created, job.folder.name, job))
        except (OSError, ValueError, TypeError) as exc:
            print(f"Skipping incomplete job {path.parent.name}: {exc}")
    if not jobs:
        raise RuntimeError("No valid waiting job found. Start cloud training in Mimicore, wait for its upload to finish, then rerun.")
    return max(jobs, key=lambda item: item[:2])[2]


def kill_tree(proc):
    # Process groups are Linux/Colab specific; descendants share this group.
    with contextlib.suppress(ProcessLookupError):
        os.killpg(proc.pid, signal.SIGKILL)
    with contextlib.suppress(subprocess.TimeoutExpired):
        proc.wait(timeout=10)


def command(args, job=None, cwd=None, timeout=1800):
    # Spool output to disk, avoiding unlimited RAM use during installation.
    with tempfile.TemporaryFile(mode="w+b") as log:
        env = dict(os.environ, MPLBACKEND="Agg")
        proc = subprocess.Popen(list(map(str, args)), cwd=cwd, stdout=log,
                                stderr=subprocess.STDOUT, start_new_session=True, env=env)
        start = time.monotonic()
        try:
            while proc.poll() is None:
                if job:
                    job.check(every=5)
                if time.monotonic() - start > timeout:
                    raise TimeoutError(f"Command timed out: {args[0]}")
                time.sleep(1)
            log.seek(0, 2)
            log.seek(max(0, log.tell() - 16000))
            output = log.read().decode("utf-8", "replace")
            if proc.returncode:
                raise RuntimeError(f"Command failed ({proc.returncode}): {args[0]}\n{output}")
            return output
        finally:
            if proc.poll() is None:
                kill_tree(proc)


class IntegrityError(RuntimeError):
    """A pinned download did not match its size or SHA-256: never retried, never used."""


def download(url, destination, job, size=None, sha256=None):
    """Fetch `url` to `destination`. With `size`/`sha256` the file must match exactly or nothing is kept (fail closed)."""
    destination.parent.mkdir(parents=True, exist_ok=True)
    part = destination.with_name(destination.name + ".part")
    for attempt in range(3):
        job.check()
        try:
            request = urllib.request.Request(url, headers={"User-Agent": "Mimicore-Colab"})
            hasher = hashlib.sha256()
            with urllib.request.urlopen(request, timeout=45) as response, part.open("wb") as f:
                total = int(response.headers.get("Content-Length") or 0)
                got = 0
                checked = time.monotonic()
                while True:
                    block = response.read(4 << 20)
                    if not block:
                        break
                    f.write(block)
                    hasher.update(block)
                    got += len(block)
                    if time.monotonic() - checked > 5:
                        job.check()
                        checked = time.monotonic()
            if not got or (total and got != total):
                raise IOError("Incomplete download")
            if size is not None and got != size:
                raise IntegrityError(f"{destination.name} has the wrong size ({got} bytes, expected {size})")
            if sha256 is not None and hasher.hexdigest() != sha256:
                raise IntegrityError(f"{destination.name} failed its SHA-256 check: the download was not the expected file")
            if destination.suffix == ".py":
                compile(part.read_bytes(), str(destination), "exec")
            os.replace(part, destination)
            return
        except (Cancelled, IntegrityError):
            part.unlink(missing_ok=True)
            raise
        except Exception:
            part.unlink(missing_ok=True)
            if attempt == 2:
                raise
            time.sleep(2 ** attempt)


def setup(job):
    job.write(state="setup", label="Preparing the cloud GPU", error=None, eta_s=None, log=[])
    if not shutil.which("nvidia-smi"):
        raise RuntimeError("Select Runtime > Change runtime type > GPU, then rerun.")
    gpu = command(["nvidia-smi", "--query-gpu=name,memory.free", "--format=csv,noheader,nounits"], job)
    name, memory = gpu.strip().splitlines()[0].rsplit(",", 1)
    available = float(memory) / 1024
    print(f"GPU: {name.strip()} — {available:.1f} GB currently free")
    if shutil.disk_usage("/content").free < 20 * 1024**3:
        raise RuntimeError("Less than 20 GB free on the Colab disk. Free space or start a fresh runtime.")
    HOME.mkdir(parents=True, exist_ok=True)
    if not (APPLIO / ".git").is_dir():
        if APPLIO.exists():
            raise RuntimeError(f"Incomplete engine checkout at {APPLIO}; use a fresh runtime.")
        command(["git", "clone", "--quiet", "https://github.com/IAHispano/Applio", APPLIO], job)
        command(["git", "checkout", "--quiet", APPLIO_COMMIT], job, APPLIO)
    actual = command(["git", "rev-parse", "HEAD"], job, APPLIO).strip()
    if actual != APPLIO_COMMIT:
        raise RuntimeError("Engine version does not match the tested pin; use a fresh runtime.")
    TOOLS.mkdir(parents=True, exist_ok=True)
    for filename in ("mimicore_train.py", "onnx_export.py", "applio_patches.py"):
        download(f"{REPO_RAW}/{filename}", TOOLS / filename, job)
    command([sys.executable, TOOLS / "applio_patches.py", APPLIO], job)
    config = APPLIO / "assets/config.json"
    if not config.exists():
        shutil.copy2(APPLIO / "assets/config_template.json", config)
    marker = HOME / "installed.json"
    version = {"applio": APPLIO_COMMIT, "tools": MIMICORE_COMMIT, "python": sys.version.split()[0]}
    installed = marker.exists() and read_json(marker) == version and PYTHON.exists()
    if not installed:
        print("Installing training dependencies in a separate Python environment...")
        shutil.rmtree(HOME / "venv", ignore_errors=True)
        command([sys.executable, "-m", "pip", "install", "--quiet", "uv"], job)
        try:
            command([sys.executable, "-m", "uv", "venv", HOME / "venv"], job)
        except Exception:
            command([sys.executable, "-m", "venv", "--without-pip", HOME / "venv"], job)
        command([sys.executable, "-m", "uv", "pip", "install", "--python", PYTHON, "--quiet",
                 "-r", "requirements.txt", "onnx", "--extra-index-url",
                 "https://download.pytorch.org/whl/cu128", "--index-strategy", "unsafe-best-match"], job, APPLIO)
        command([PYTHON, "core.py", "prerequisites", "--no-pretraineds-hifigan", "--models", "--no-exe"], job, APPLIO)
    command([PYTHON, "-c", "import torch,onnx; assert torch.cuda.is_available(), 'PyTorch cannot use this GPU'; print(torch.cuda.get_device_name(0))"], job)
    if job.base == "titan":
        target = APPLIO / "rvc/models/pretraineds/custom/TITAN_Medium_recommended_for_speech"
        for kind in ("G", "D"):
            path = target / f"{kind}-f048k-TITAN-Medium.pth"
            want_size, want_sha = TITAN_FILES[kind]
            if not (path.is_file() and path.stat().st_size == want_size and digest(path) == want_sha):
                print(f"Downloading base voice model ({kind})...")
                download(TITAN_URL.format(kind), path, job, size=want_size, sha256=want_sha)
    marker.write_text(json.dumps(version), encoding="utf-8")
    return available


def sweep_resume(job):
    """Remove checkpoint backups of jobs deleted in Mimicore. Refuses when any job.json is unreadable (mid-write)."""
    root = job.folder.parent
    live = {job.name}  # a flaky Drive listing must never make the job being trained look deleted
    for jf in root.glob("*/job.json"):
        try:
            live.add(Job(jf.parent).name)
        except (OSError, ValueError, TypeError):
            return
    for old in (root / "_resume").glob("*"):
        if old.is_dir() and old.name not in live:
            shutil.rmtree(old, ignore_errors=True)


def prepare(job):
    local = WORK / job.name
    dataset, runs, out = local / "dataset", local / "runs", local / "out"
    dataset.mkdir(parents=True, exist_ok=True)
    folder = job.folder / "recordings"
    drive_alt = job.folder.parent / "Recordings"
    custom_drive = Path("/content/drive/MyDrive/Mimicore/Recordings")
    custom_drive_alt = Path("/content/drive/MyDrive/Recordings")
    candidate_folders = [folder, drive_alt, custom_drive, custom_drive_alt]
    recordings = []
    for cand in candidate_folders:
        if cand.is_dir():
            found = sorted(p for p in cand.iterdir()
                           if p.is_file() and not p.is_symlink() and not p.name.startswith(".")
                           and p.suffix.lower() in AUDIO_EXTENSIONS)
            if found:
                recordings = found
                if cand != folder:
                    print(f"Direct Google Drive Import: found {len(recordings)} audio recordings in {cand}")
                break
    if not recordings:
        if folder.is_dir():
            recordings = sorted(p for p in folder.iterdir()
                                if p.is_file() and not p.is_symlink() and not p.name.startswith("."))
        if not recordings:
            raise RuntimeError("No supported audio recordings found in this job or in Google Drive (MyDrive/Mimicore/Recordings).")
    odd = [p.name for p in recordings if p.suffix.lower() not in AUDIO_EXTENSIONS]
    if odd:
        print(f"Note: unusual file type(s) will be tried anyway: {', '.join(odd[:5])}")
    required = sum(p.stat().st_size for p in recordings)
    if shutil.disk_usage("/content").free < required * 4 + 10 * 1024**3:
        raise RuntimeError("Not enough local disk space for this dataset and checkpoints.")
    shutil.rmtree(dataset)
    dataset.mkdir()
    fingerprint = hashlib.sha256(f"{job.base}:{APPLIO_COMMIT}:{MIMICORE_COMMIT}".encode())
    print(f"Copying and fingerprinting {len(recordings)} recording(s)...")
    for i, source in enumerate(recordings):
        job.check()
        if not source.stat().st_size:
            raise ValueError(f"Empty recording: {source.name}")
        destination = dataset / f"{i:05}{source.suffix.lower()}"
        before = signature(source)
        shutil.copy2(source, destination)
        if signature(source) != before or destination.stat().st_size != before[0]:
            raise IOError("A recording changed while copying. Wait for the app upload to finish.")
        fingerprint.update(json.dumps([source.name, digest(destination)]).encode())
    identity = fingerprint.hexdigest()
    resume = job.folder.parent / "_resume" / job.name / identity
    runs.mkdir(exist_ok=True)
    return local, dataset, runs, out, resume, identity


def signature(path):
    stat = path.stat()
    return stat.st_size, stat.st_mtime_ns


def validate_pair(folder):
    # Run in the training environment, load one state at a time on CPU, and
    # require matching epochs plus optimizer state before accepting a backup.
    script = """
import json, pathlib, sys, torch
root = pathlib.Path(sys.argv[1])
epochs = []
for prefix in ('G_', 'D_'):
    files = list(root.glob(prefix + '*.pth'))
    assert len(files) == 1, 'Expected one generator/discriminator checkpoint'
    state = torch.load(files[0], map_location='cpu', weights_only=True)
    assert isinstance(state.get('model'), dict) and isinstance(state.get('optimizer'), dict)
    epochs.append(int(state['iteration']))
    del state
assert epochs[0] == epochs[1] and epochs[0] > 0, 'Mismatched checkpoint epochs'
print(json.dumps({'epoch': epochs[0]}))
"""
    return json.loads(command([PYTHON, "-c", script, folder], timeout=180).strip())["epoch"]


class Backups:
    """Publish complete, hashed generations; incomplete copies are never restored."""
    def __init__(self, run_dir, resume, identity):
        self.run_dir, self.resume, self.identity = run_dir, resume, identity
        self.lock = threading.Lock()
        self.last_signature = None

    def save(self):
        with self.lock:
            generators = list(self.run_dir.glob("G_*.pth"))
            discriminators = list(self.run_dir.glob("D_*.pth"))
            if not generators or not discriminators:
                return
            # The pinned trainer saves only its latest G/D pair.
            pair = [max(generators, key=lambda p: p.stat().st_mtime_ns),
                    max(discriminators, key=lambda p: p.stat().st_mtime_ns)]
            before = [signature(p) for p in pair]
            if before == self.last_signature:
                return
            with tempfile.TemporaryDirectory(dir=self.run_dir.parent) as tmp:
                stage = Path(tmp)
                paths = pair + list(self.run_dir.glob("*_*e_*s.pth"))
                paths += [self.run_dir / n for n in ("config.json", "model_info.json", "training_data.json")
                          if (self.run_dir / n).is_file()]
                for src in paths:
                    original = signature(src)
                    shutil.copy2(src, stage / src.name)
                    if signature(src) != original:
                        raise IOError("Checkpoint changed during backup; will retry")
                if [signature(p) for p in pair] != before:
                    raise IOError("Checkpoint pair changed during backup; will retry")
                epoch = validate_pair(stage)
                # TensorBoard logs support the trainer's existing checkpoint selection.
                if (self.run_dir / "eval").is_dir():
                    shutil.copytree(self.run_dir / "eval", stage / "eval")
                files = {str(p.relative_to(stage).as_posix()): digest(p)
                         for p in stage.rglob("*") if p.is_file()}
                manifest = {"identity": self.identity, "epoch": epoch, "files": files}
                self.resume.mkdir(parents=True, exist_ok=True)
                generation = self.resume / f"snapshot-{time.time_ns()}-{uuid.uuid4().hex[:8]}"
                shutil.copytree(stage, generation)
                for name, expected in files.items():
                    if digest(generation / name) != expected:
                        raise IOError("Drive checkpoint verification failed")
                # Commit record written last. A torn JSON record is ignored on restore.
                (generation / "complete.json").write_text(json.dumps(manifest), encoding="utf-8")
                self.last_signature = before
                committed = sorted(p.parent for p in self.resume.glob("snapshot-*/complete.json"))
                for old in committed[:-2]:
                    shutil.rmtree(old)
                # copies torn by a crash or a failed verification never get a complete.json: do not let them pile up
                # in the user's Drive quota
                for torn in self.resume.glob("snapshot-*"):
                    if torn.is_dir() and not (torn / "complete.json").exists() and torn != generation:
                        shutil.rmtree(torn, ignore_errors=True)
                print(f"Checkpoint backed up: epoch {epoch}")

    def restore(self):
        # Start from verified backups, not potentially torn local files after Ctrl-C.
        shutil.rmtree(self.run_dir, ignore_errors=True)
        for record in sorted(self.resume.glob("snapshot-*/complete.json"), reverse=True):
            try:
                manifest = read_json(record)
                if manifest.get("identity") != self.identity:
                    continue
                with tempfile.TemporaryDirectory(dir=self.run_dir.parent) as tmp:
                    stage = Path(tmp)
                    for name, expected in manifest["files"].items():
                        rel = Path(name)
                        if rel.is_absolute() or ".." in rel.parts or "\\" in name:
                            raise ValueError("Unsafe backup path")
                        dest = stage / rel
                        dest.parent.mkdir(parents=True, exist_ok=True)
                        shutil.copy2(record.parent / rel, dest)
                        if digest(dest) != expected:
                            raise ValueError("Backup checksum mismatch")
                    validate_pair(stage)
                    shutil.copytree(stage, self.run_dir)
                print(f"Restored verified backup at epoch {manifest['epoch']}")
                self.last_signature = None
                return True
            except Exception as exc:
                shutil.rmtree(self.run_dir, ignore_errors=True)
                print(f"Skipping unusable backup: {exc}")
        print("No verified resume checkpoint; starting from the base model.")
        return False

    def loop(self, stop):
        while not stop.wait(BACKUP_SECONDS):
            try:
                self.save()
            except Exception as exc:
                print(f"Backup deferred: {exc}")


def train_once(job, dataset, runs, out, batch, backups):
    job.check()
    shutil.rmtree(out, ignore_errors=True)
    job.try_write(state="training", step=1, steps=5, label="Cleaning & slicing audio",
                  epoch=0, epochs=job.epochs, batch_used=batch, eta_s=None, error=None)
    args = [PYTHON, "-u", TOOLS / "mimicore_train.py", "--name", job.name,
            "--dataset", dataset, "--runs", runs, "--out", out,
            "--epochs", job.epochs, "--batch", batch, "--basis", job.base, "--tools", TOOLS]
    proc = subprocess.Popen(list(map(str, args)), cwd=APPLIO, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, encoding="utf-8",
                            errors="replace", bufsize=1, start_new_session=True,
                            env={**os.environ, "PYTHONUNBUFFERED": "1", "MPLBACKEND": "Agg"})
    stop = threading.Event()
    monitor_errors = []

    def monitor():
        while not stop.wait(5):
            try:
                job.check()
            except Cancelled as exc:
                monitor_errors.append(exc)
                kill_tree(proc)
                return

    watcher = threading.Thread(target=monitor, daemon=True)
    backup_thread = threading.Thread(target=backups.loop, args=(stop,), daemon=True)
    watcher.start()
    backup_thread.start()
    tail = collections.deque(maxlen=60)
    oom = False
    start_epoch = start_time = None
    last_write = 0
    try:
        for raw in proc.stdout:
            line = raw.rstrip()
            tail.append(line)
            oom = oom or any(token in line.lower() for token in OOM_TEXT)
            if line.startswith("@@"):
                try:
                    event = json.loads(line[2:])
                    if not isinstance(event, dict):
                        raise ValueError("Invalid progress event")
                    if "label" in event:
                        job.update(label=str(event["label"])[:300])
                    if "step" in event:
                        job.update(step=int(event["step"]), steps=int(event.get("of", 5)), eta_s=None)
                        print(f"Step {event['step']}/5: {event.get('label', '')}")
                    if "epoch" in event:
                        epoch = integer(event["epoch"], "progress epoch", 0, job.epochs)
                        now = time.monotonic()
                        if start_time is None:
                            start_epoch, start_time = epoch, now
                        elif epoch > start_epoch:
                            job.update(eta_s=max(0, int((now - start_time) * (job.epochs - epoch) / (epoch - start_epoch))))
                        job.update(epoch=epoch)
                        if epoch % 10 == 0:
                            print(f"Epoch {epoch}/{job.epochs}")
                    if time.monotonic() - last_write > 20 or "step" in event:
                        job.try_write()
                        last_write = time.monotonic()
                except (ValueError, TypeError, KeyError) as exc:
                    print(f"Ignored malformed progress event: {exc}")
            elif "error" in line.lower() or "traceback" in line.lower():
                print(line)
        proc.wait()
    finally:
        stop.set()
        # Always kill descendants too, including after the wrapper exits.
        kill_tree(proc)
        proc.stdout.close()
        watcher.join()
        backup_thread.join()
        try:
            backups.save()
        except Exception as exc:
            print(f"Final backup unavailable; previous verified backup retained: {exc}")
    if monitor_errors:
        raise monitor_errors[0]
    job.check()
    if proc.returncode:
        job.update(log=list(tail)[-30:])
        return False, oom, "\n".join(tail)
    return True, False, ""


def train(job, vram):
    local, dataset, runs, out, resume, identity = prepare(job)
    sweep_resume(job)
    backups = Backups(runs / job.name, resume, identity)
    backups.restore()
    automatic = 16 if vram >= 20 else 12 if vram >= 11 else 8 if vram >= 7 else 4
    batch = job.batch or automatic
    saved = job.status.get("retry_batch")
    if saved is not None:
        batch = min(batch, integer(saved, "retry batch", 1, 128))
    for attempt in range(MAX_OOM_RETRIES + 1):
        ok, oom, log = train_once(job, dataset, runs, out, batch, backups)
        if ok:
            return out, resume
        if not oom or batch == 1:
            raise RuntimeError(log or "Training failed without diagnostic output")
        batch = max(1, batch // 2)
        job.write(retry_batch=batch, eta_s=None, label=f"GPU memory exhausted; reducing batch to {batch}")
        if attempt == MAX_OOM_RETRIES:
            raise RuntimeError(f"GPU memory remained insufficient after automatic retries. Rerun to try batch {batch}.\n{log}")
        print(f"GPU memory exhausted. Retrying with batch {batch}.")
        backups.restore()


def deliver(job, out):
    job.check()
    for name in ("voice.onnx", "index.npy", "meta.json"):
        path = out / name
        if not path.is_file() or path.stat().st_size == 0:
            raise RuntimeError(f"Export is incomplete: missing or empty {name}")
    read_json(out / "meta.json")
    validation = "import sys,onnx,numpy as np; from pathlib import Path; p=Path(sys.argv[1]); onnx.checker.check_model(str(p/'voice.onnx')); a=np.load(p/'index.npy',mmap_mode='r',allow_pickle=False); assert a.ndim==2 and a.size>0, 'Invalid voice index'"
    command([PYTHON, "-c", validation, out], job, timeout=300)
    job.write(state="uploading", label="Verifying and copying your voice to Drive", eta_s=None)
    archive = out.parent / "voice.tar"
    with tarfile.open(archive, "w") as tar:
        for path in sorted(out.rglob("*")):
            if path.is_symlink():
                raise RuntimeError("Unexpected symbolic link in voice export")
            if path.is_file():
                tar.add(path, arcname=path.relative_to(out).as_posix(), recursive=False)
    expected = digest(archive)
    destination = job.folder / "voice.tar"
    # Do not replace this app-created file: Mimicore depends on its Drive file ID.
    with archive.open("rb") as src, destination.open("r+b") as dst:
        dst.seek(0)
        while True:
            job.check()
            block = src.read(8 << 20)
            if not block:
                break
            dst.write(block)
        dst.truncate()
        dst.flush()
        os.fsync(dst.fileno())
    if destination.stat().st_size != archive.stat().st_size or digest(destination) != expected:
        raise IOError("Uploaded voice verification failed; recordings and backups were kept")
    job.check()
    if not job.write(state="done", label="Done", error=None, eta_s=0,
                     voice_bytes=archive.stat().st_size, voice_sha256=expected):
        raise Cancelled("Job was stopped before completion could be published.")
    # Backups are deliberately retained until the job is explicitly cleaned up.
    # No cleanup of unrelated jobs or recordings by default.
    # The voice is verified on Drive: its checkpoints are no longer needed and would fill the user's quota.
    shutil.rmtree(job.folder.parent / "_resume" / job.name, ignore_errors=True)
    if REMOVE_RECORDINGS_AFTER_SUCCESS and not job.spec.get("keep_files"):
        shutil.rmtree(job.folder / "recordings", ignore_errors=True)


def main():
    job = None
    stop = threading.Event()
    hb = None
    success = False
    try:
        job = find_job()
        job.check()
        hb = threading.Thread(target=heartbeat, args=(job, stop), daemon=True)
        hb.start()
        who = str(job.spec.get("name") or "your voice")
        banner(f"Training “{who}”. Keep this tab open.")
        vram = setup(job)
        out, _ = train(job, vram)
        deliver(job, out)
        success = True
    except Cancelled as exc:
        if job:
            with contextlib.suppress(Exception):
                job.write(state="cancelled", label="Cancelled", eta_s=None)
        banner(str(exc), "#b45309")
    except KeyboardInterrupt:
        if job:
            with contextlib.suppress(Exception):
                job.write(state="failed", label="Interrupted; rerun to restore backup", error="Stopped from Colab", eta_s=None)
        banner("Interrupted. Rerun to restore the latest verified backup, if available.", "#b45309")
    except Exception as exc:
        traceback.print_exc()
        lines = [line for line in str(exc).splitlines() if line.strip()]
        message = lines[-1] if lines else type(exc).__name__
        if job:
            with contextlib.suppress(Exception):
                job.write(state="failed", label="Stopped", error=message[:500], log=lines[-30:], eta_s=None)
        banner(f"Training stopped: {message}. Fix the reported problem, then rerun. Verified backups are retained.", "#b91c1c")
    finally:
        stop.set()
        if hb:
            hb.join()
    if success:
        # Flush/unmount failures must not change a completed job to failed, and
        # must not release the runtime before Drive has finished flushing.
        try:
            from google.colab import drive
            drive.flush_and_unmount()
        except Exception as exc:
            banner(f"Voice export completed, but Drive flush failed: {exc}. Keep this runtime open and check Mimicore before disconnecting.", "#b45309")
            return
        banner(f"Done! “{who}” is ready for Mimicore to install.", "#059669")
        if RELEASE_GPU_AFTER_SUCCESS:
            try:
                from google.colab import runtime
                runtime.unassign()
            except Exception as exc:
                print(f"Voice is complete. Disconnect the runtime manually to release the GPU: {exc}")


def run_notebook():
    # Prevent two cells in this runtime from modifying the same training engine.
    # This does not lock another Colab runtime; run only one session per job.
    import fcntl
    HOME.mkdir(parents=True, exist_ok=True)
    with (HOME / "session.lock").open("a+") as lock:
        try:
            fcntl.flock(lock, fcntl.LOCK_EX | fcntl.LOCK_NB)
        except BlockingIOError:
            banner("A training cell is already running in this runtime. Stop it before starting another.", "#b45309")
            return
        try:
            main()
        finally:
            fcntl.flock(lock, fcntl.LOCK_UN)


if __name__ == "__main__":
    run_notebook()
